# Refua Showcase Notebook: SARS-CoV-2 Mpro Ligand Triage

> **Audience:** beginner to advanced.  
> **Goal:** demonstrate a practical end-to-end Refua workflow for protein-ligand hypothesis triage with rich notebook visualization.

## Why this is a good showcase
- Uses a **real structural benchmark**: SARS-CoV-2 main protease (Mpro) with PF-07321332 / nirmatrelvir context (PDB 7VH8).
- Exercises multiple Refua capabilities in one flow: `SM`, `Protein`, `Complex`, `request_affinity`, `fold`, and rich widget rendering.
- Keeps the workflow realistic: inspect chemistry first, then build/fold, then interpret affinity outputs for **relative ranking**.

## What you will do
1. Define an Mpro target sequence and nirmatrelvir ligand SMILES.
2. Inspect molecule-level views (protein + ligand widgets).
3. Build and fold a complex with an affinity request.
4. Read the result with practical interpretation guardrails.

## Feature map (what this notebook showcases)
| Refua object/API | Notebook role | Why it matters |
|---|---|---|
| `SM(...)` | Ligand object with chemistry/ADMET display | Fast molecular sanity check before expensive runs |
| `Protein(...)` | Target macromolecule object | Explicit, reproducible target definition |
| `Complex([...])` | Modeling request container | Makes setups versionable and comparable |
| `.request_affinity(...)` | Adds affinity-oriented output request | Supports candidate prioritization workflows |
| `.fold()` | Executes design/folding pipeline | Produces structural and scoring hypotheses for triage |


In [ ]:
%load_ext refua_notebook
import refua_notebook

refua_notebook.activate()


In [ ]:
from refua import Complex, Protein, SM


## Biological setup (plain language)

SARS-CoV-2 Mpro is a viral protease required for processing viral polyproteins.
It has been a high-priority antiviral target because inhibiting this enzyme can block viral replication steps.

Here we model a known inhibitor scaffold context using PF-07321332 (nirmatrelvir chemical identity) against Mpro.

| Design input | Choice in this notebook | Why |
|---|---|---|
| Target sequence | PDB 7VH8 Mpro chain A sequence | Experimentally solved target-ligand benchmark context |
| Ligand | Nirmatrelvir SMILES (PubChem CID 155903259) | Real antiviral protease inhibitor chemistry |
| Optional comparator | Caffeine SMILES | Lightweight non-matched control branch for workflow testing |


In [ ]:
MPRO_SEQUENCE_7VH8 = (
    "SGFRKMAFPSGKVEGCMVQVTCGTTTLNGLWLDDVVYCPRHVICTSEDMLNPNYEDLLIRKSNHNFLVQAGNVQLRVIGHSMQNCVLKLKVDTANPKTPKYKFVRIQPGQTFSVLACYNGSPSGVYQCAMRPNFTIKGSFLNGSCGSVGFNIDYDCVSFCYMHHMELPTGVHAGTDLEGNFYGPFVDRQTAQAAGTDTTITVNVLAWLYAAVINGDRWFLNRFTTTLNDFNLVAMKYNYEPLTQDHVDILGPLSAQTGIAVLDMCASLKELLQNGMNGRTILGSALLEDEFTPFDVVRQCSGVTFQ"
)

# PubChem CID 155903259, isomeric SMILES for nirmatrelvir.
NIRMATRELVIR_SMILES = "CC1([C@@H]2[C@H]1[C@H](N(C2)C(=O)[C@H](C(C)(C)C)NC(=O)C(F)(F)F)C(=O)N[C@@H](C[C@@H]3CCNC3=O)C#N)C"

# Optional comparison branch: not expected to be a matched Mpro inhibitor.
CAFFEINE_SMILES = "Cn1cnc2c1c(=O)n(C)c(=O)n2C"

protein = Protein(MPRO_SEQUENCE_7VH8, ids="A")
ligand = SM(NIRMATRELVIR_SMILES)


## Inspect molecule-level widgets first

Before folding any complex, it is good practice to sanity-check inputs visually.

For `protein`, you can confirm sequence-level identity/context.
For `ligand`, you can quickly inspect structure and computed properties/ADMET panels.

> Practical habit: catch data-entry mistakes early here (sequence truncation, malformed SMILES, unexpected stereochemistry) before spending compute.


In [ ]:
protein


In [ ]:
ligand


## Build a reproducible complex request

`Complex([protein, ligand])` captures the exact modeling setup.

Why this matters for real projects:
- You can version this setup in Git.
- You can clone/modify it for comparison branches.
- You can keep naming conventions (`name=...`) consistent for downstream tracking.


In [ ]:
complex_spec = Complex([protein, ligand], name="mpro_nirmatrelvir_showcase")

complex_spec


## Fold + affinity request

`request_affinity(ligand)` asks Refua to include affinity-oriented outputs for this ligand.
Then `.fold()` runs the modeling pipeline.

What to read after this cell:
- `complex_spec`: updated rich view of the requested/available complex context.
- `result.affinity`: relative signal for ranking hypotheses.

> Guardrail: affinity outputs are best used for **relative prioritization** across candidates, not standalone claims of true experimental potency.


In [ ]:
result = complex_spec.request_affinity(ligand).fold()

complex_spec


In [ ]:
result.affinity


## Optional follow-up: quick comparison branch

Use this pattern to compare multiple ligands under the same target setup:

```python
alt_ligand = SM(CAFFEINE_SMILES)
alt_spec = Complex([protein, alt_ligand], name="mpro_caffeine_control")
alt_result = alt_spec.request_affinity(alt_ligand).fold()

alt_ligand
alt_spec
alt_result.affinity
```

Then compare both runs side-by-side and keep only candidates that look consistently strong across structural and affinity signals.

## Science references

- RCSB PDB 7VH8 (SARS-CoV-2 Mpro with PF-07321332): https://www.rcsb.org/structure/7VH8
- RCSB FASTA 7VH8 (exact chain sequence used here): https://www.rcsb.org/fasta/entry/7VH8/display
- Zhao et al., *Protein & Cell* (2022), Mpro-PF-07321332 structure (PMID: 34687004): https://pubmed.ncbi.nlm.nih.gov/34687004/
- PubChem CID 155903259 (nirmatrelvir SMILES metadata): https://pubchem.ncbi.nlm.nih.gov/compound/155903259
- FDA preclinical research basics (why computational results need experimental validation): https://www.fda.gov/patients/drug-development-process/step-2-preclinical-research
